# Bab 6: Building Your First Deep Neural Network — Introduction to Backpropagation

Notebook ini merangkum Bab 6 buku *Grokking Deep Learning* karya Andrew W. Trask, bab penutup dari seri yang diminta untuk tugas ini. Bab ini memperkenalkan **backpropagation**: cara mengalirkan sinyal error ke belakang melewati banyak layer, yang menjadi dasar dari semua deep neural network modern. Konteksnya dibangun lewat studi kasus "streetlight problem".

Sesuai instruksi tugas, notebook ini hanya memuat kode dan penjelasan tanpa perlu dijalankan.

## 1. Studi Kasus: Masalah Lampu Lalu Lintas (Streetlight Problem)

Bayangkan kita berdiri di perempatan jalan di negara asing dan tidak tahu cara membaca lampu lalu lintasnya. Kita bisa mempelajari pola ini dengan mengamati kombinasi lampu mana yang berkorelasi dengan orang-orang yang menyeberang (WALK) atau berhenti (STOP). Setelah mengamati enam kombinasi lampu, ternyata hanya **lampu tengah** yang berkorelasi sempurna dengan aman-tidaknya menyeberang.

Inilah yang akan dilatih untuk dipelajari oleh neural network: menemukan korelasi antara pola lampu (data yang kita tahu) dan aman-tidaknya menyeberang (data yang ingin kita ketahui).

## 2. Menyiapkan Data: Dari Pola Lampu ke Matrix

Pola lampu lalu lintas diterjemahkan menjadi matrix berisi 1 dan 0: setiap baris mewakili satu pengamatan, setiap kolom mewakili satu lampu. Prinsip pentingnya: ada banyak matrix berbeda (bahkan hasil perkalian skalar dari satu sama lain) yang tetap merepresentasikan pola dasar yang sama. Yang penting bukan angka persisnya, tapi **pola** yang terkandung di dalamnya.

In [1]:
import numpy as np

streetlights = np.array([[1, 0, 1],
                          [0, 1, 1],
                          [0, 0, 1],
                          [1, 1, 1],
                          [0, 1, 1],
                          [1, 0, 1]])

walk_vs_stop = np.array([[0],
                          [1],
                          [0],
                          [1],
                          [1],
                          [0]])

Perhatikan dua operasi NumPy yang berguna: **elementwise multiplication/addition** antar dua vektor, serta **vector-scalar multiplication/addition**.

In [2]:
a = np.array([0, 1, 2, 1])
b = np.array([2, 2, 2, 3])

print(a * b)    # elementwise multiplication
print(a + b)    # elementwise addition
print(a * 0.5)  # vector-scalar multiplication
print(a + 0.5)  # vector-scalar addition

[0 2 4 3]
[2 3 4 4]
[0.  0.5 1.  0.5]
[0.5 1.5 2.5 1.5]


## 3. Melatih Network untuk Satu Pola Lampu

Sebagai permulaan, mari latih network hanya dengan satu baris data (satu pola lampu) terlebih dahulu. Ini mirip dengan network satu-input-banyak-weight yang sudah dibahas di Bab 3-4.

In [3]:
weights = np.array([0.5, 0.48, -0.7])
alpha = 0.1

input = streetlights[0]  # [1, 0, 1], artinya STOP (0)
goal_prediction = walk_vs_stop[0]

for iteration in range(20):
    prediction = input.dot(weights)
    error = (goal_prediction - prediction) ** 2
    delta = prediction - goal_prediction
    weights = weights - (alpha * (input * delta))
    print("Error:" + str(error) + " Prediction:" + str(prediction))

Error:[0.04] Prediction:-0.19999999999999996
Error:[0.0256] Prediction:-0.15999999999999992
Error:[0.016384] Prediction:-0.1279999999999999
Error:[0.01048576] Prediction:-0.10239999999999982
Error:[0.00671089] Prediction:-0.08191999999999977
Error:[0.00429497] Prediction:-0.06553599999999982
Error:[0.00274878] Prediction:-0.05242879999999994
Error:[0.00175922] Prediction:-0.04194304000000004
Error:[0.0011259] Prediction:-0.03355443200000008
Error:[0.00072058] Prediction:-0.02684354560000002
Error:[0.00046117] Prediction:-0.021474836479999926
Error:[0.00029515] Prediction:-0.01717986918399994
Error:[0.00018889] Prediction:-0.013743895347199997
Error:[0.00012089] Prediction:-0.010995116277759953
Error:[7.73712525e-05] Prediction:-0.008796093022207963
Error:[4.95176016e-05] Prediction:-0.007036874417766459
Error:[3.1691265e-05] Prediction:-0.0056294995342132115
Error:[2.02824096e-05] Prediction:-0.004503599627370569
Error:[1.29807421e-05] Prediction:-0.003602879701896544
Error:[8.30767497

## 4. Melatih Network untuk Seluruh Dataset

Satu pola lampu saja tidak cukup, kita ingin network mengenali keenam pola lampu sekaligus. Caranya: latih network pada semua baris data, satu per satu, berulang-ulang (banyak iterasi/epoch).

In [4]:
weights = np.array([0.5, 0.48, -0.7])
alpha = 0.1

for iteration in range(40):
    error_for_all_lights = 0
    for row_index in range(len(walk_vs_stop)):
        input = streetlights[row_index]
        goal_prediction = walk_vs_stop[row_index]

        prediction = input.dot(weights)
        error = (goal_prediction - prediction) ** 2
        error_for_all_lights += error

        delta = prediction - goal_prediction
        weights = weights - (alpha * (input * delta))
        print("Prediction:" + str(prediction))

    print("Error:" + str(error_for_all_lights) + "\n")

# Error akan terus mengecil dari sekitar 2.65 di awal menuju sangat kecil
# (orde 0.0005) setelah 40 iterasi penuh

Prediction:-0.19999999999999996
Prediction:-0.19999999999999996
Prediction:-0.5599999999999999
Prediction:0.6160000000000001
Prediction:0.17279999999999995
Prediction:0.17552
Error:[2.65612311]

Prediction:0.14041599999999999
Prediction:0.3066464
Prediction:-0.34513824
Prediction:1.006637344
Prediction:0.4785034751999999
Prediction:0.26700416768
Error:[0.96287018]

Prediction:0.213603334144
Prediction:0.5347420299776
Prediction:-0.26067345110016
Prediction:1.1319428845096962
Prediction:0.6274723921901568
Prediction:0.25433999330650114
Error:[0.55091659]

Prediction:0.20347199464520088
Prediction:0.6561967149569552
Prediction:-0.221948503950995
Prediction:1.166258650532124
Prediction:0.7139004922542389
Prediction:0.21471099528371604
Error:[0.36445837]

Prediction:0.17176879622697283
Prediction:0.7324724146523222
Prediction:-0.19966478845083285
Prediction:1.1697769945341199
Prediction:0.7719890116601171
Prediction:0.17297997428859369
Error:[0.25167687]

Prediction:0.13838397943087496
Pre

### Full, Batch, dan Stochastic Gradient Descent

Ada tiga varian umum cara melatih network memakai seluruh dataset:

- **Stochastic gradient descent**: mengupdate weight setiap kali selesai memproses **satu** contoh data (seperti kode di atas). Network memprediksi satu baris, menghitung weight_delta, mengupdate weight, lalu lanjut ke baris berikutnya.
- **(Full) gradient descent**: mengupdate weight hanya **sekali** per seluruh dataset, memakai rata-rata weight_delta dari semua contoh data.
- **Batch gradient descent**: jalan tengah antara keduanya, mengupdate weight setiap setelah memproses sejumlah `n` contoh (biasanya antara 8 dan 256), bukan satu per satu atau seluruh dataset sekaligus.

## 5. Neural Network Belajar Korelasi

Setelah network dilatih, lihat posisi akhir weight-nya: weight untuk lampu tengah mendekati 1.0, sementara weight untuk lampu kiri dan kanan mendekati 0. Secara sederhana, semua proses iteratif gradient descent ini sebenarnya hanya melakukan satu hal: mengidentifikasi **korelasi** antara input dan output.

Korelasi muncul di mana weight bernilai tinggi; ketidakberkaitan (randomness) muncul di mana weight mendekati 0. Ini terjadi karena setiap contoh data memberi "tekanan" naik atau turun pada tiap weight, dan rata-rata tekanan itulah yang menentukan arah akhir tiap weight.

### Dari Mana Tekanan Ini Berasal?

Tekanan berasal langsung dari data. Setiap node pada dasarnya mencoba memprediksi output secara independen, dan satu-satunya komunikasi antar weight adalah lewat error yang dibagi bersama (shared error). Perhatikan contoh data pertama: karena input lampu tengah adalah 0, weight lampu tengah sama sekali tidak relevan untuk prediksi itu (berapa pun nilainya, akan dikalikan 0). Jadi error pada contoh itu hanya bisa diatribusikan ke weight lampu kiri dan kanan.

Dengan menjumlahkan tekanan dari semua contoh data, weight lampu tengah secara konsisten mendapat tekanan positif (menuju 1), sementara weight lainnya mendapat tekanan yang saling membatalkan atau cenderung negatif (menuju 0).

## 6. Edge Case: Overfitting

Terkadang korelasi sempurna bisa muncul **secara tidak sengaja**. Misalnya, kalau weight lampu kiri 0.5 dan weight lampu kanan -0.5, prediksinya bisa pas menjadi 0 untuk contoh tertentu, padahal network sama sekali belum benar-benar "memahami" pola lampu yang sesungguhnya. Fenomena ini disebut **overfitting**.

Ini adalah kelemahan terbesar deep learning: karena error dibagi bersama semua weight, kalau suatu konfigurasi weight secara kebetulan menghasilkan korelasi sempurna (error = 0) tanpa benar-benar memberi bobot terbesar ke input terbaik, network akan berhenti belajar di titik itu. Untungnya, keberadaan banyak contoh data lain biasanya mencegah network terjebak di konfigurasi "kebetulan" semacam ini, karena contoh-contoh lain akan mengganggu ekuilibrium yang salah tersebut.

Pelajaran intinya: tantangan terbesar dalam deep learning adalah membuat network **generalisasi**, bukan sekadar **menghafal** (memorize) data training.

## 7. Edge Case: Tekanan yang Saling Bertentangan

Kadang satu kolom input punya jumlah tekanan naik dan turun yang kelihatannya seimbang, tapi tetap berhasil didorong ke 0 oleh network. Ini terjadi karena begitu weight lain (yang punya sinyal lebih jelas) mulai konvergen ke posisi optimalnya, mereka menyerap sebagian besar korelasi yang dibutuhkan, sehingga error yang tersisa untuk weight yang "ambigu" ini jadi semakin jelas arahnya sendiri dan akhirnya terdorong ke 0.

Namun, kalau tidak ada satu pun kolom input yang berkorelasi dengan output (semua weight mendapat tekanan naik-turun yang benar-benar seimbang), network akan kesulitan belajar apa pun. Konsep **regularization** (dibahas lebih lanjut di bab-bab berikutnya) membantu mengatasi ini dengan secara aktif mendorong weight yang tidak punya korelasi jelas menuju 0 lebih cepat.

## 8. Belajar Korelasi Tidak Langsung

Bagaimana jika dataset-nya benar-benar tidak punya korelasi langsung antara kolom input mana pun dengan output? Solusinya: buat **dataset perantara (intermediate dataset)** yang memang punya korelasi dengan output, dengan menumpuk dua neural network. Network pertama (`layer_0` ke `layer_1`) menghasilkan representasi perantara, dan network kedua (`layer_1` ke `layer_2`) memakai representasi itu untuk memprediksi output akhir.

Ini pada dasarnya sama dengan network bertumpuk (stacked network) yang sudah disinggung di Bab 3, hanya sekarang kita akan membahas bagaimana cara melatihnya secara utuh, termasuk bagaimana layer tengah (`layer_1`) ikut diupdate.

## 9. Backpropagation: Atribusi Error Jarak Jauh

Pertanyaan kuncinya: weight antara `layer_0` dan `layer_1` memakai ukuran error apa untuk belajar? Jawabannya: kita perlu menghitung `delta` di `layer_1`, dengan cara mengambil `delta` di `layer_2` dan mengalikannya dengan weight yang menghubungkan kedua layer tersebut (`weights_1_2`). Node di `layer_1` dengan weight yang lebih besar ke `layer_2` dianggap berkontribusi lebih besar terhadap error, dan sebaliknya.

Proses menyebarkan sinyal delta ke belakang seperti inilah yang disebut **backpropagation**: pada dasarnya logika prediksi (forward propagation), hanya dijalankan terbalik untuk mendistribusikan error.

## 10. Linear vs Nonlinear: Masalah yang Harus Dipecahkan

Ada satu masalah mendasar: dua perkalian berurutan secara matematis bisa selalu disederhanakan jadi satu perkalian saja (`1 * 10 * 2 = 100` sama hasilnya dengan `5 * 20 = 100`). Artinya, **menumpuk dua network linear seperti ini tidak memberi kekuatan tambahan apa pun** dibanding satu network saja, karena dua weighted sum berurutan cuma versi yang lebih mahal dari satu weighted sum.

Konsekuensinya: tanpa ada perubahan, `layer_1` hanya akan selalu "berlangganan" sejumlah persentase tetap dari korelasi input, baik 100%, 50%, atau proporsi tertentu lainnya, dan tidak pernah bisa punya korelasi miliknya sendiri yang bersifat kondisional.

### Solusi: Korelasi Kondisional Lewat Nonlinearitas

Triknya sederhana: **matikan node (set jadi 0) setiap kali nilainya akan negatif**. Dengan begitu, node tersebut punya korelasi nol terhadap input kapan pun nilainya negatif, dan baru "menyala" (berkorelasi) ketika nilainya positif. Ini memungkinkan node tengah berkata, misalnya, "aku berkorelasi penuh dengan input kiri, tapi hanya kalau input kanan sedang mati."

Logika "kalau nilainya akan negatif, jadikan 0" ini disebut **nonlinearitas**, dan versi yang dipakai di buku ini adalah yang paling sederhana dan populer: **ReLU** (Rectified Linear Unit).

In [5]:
def relu(x):
    return (x > 0) * x  # Mengembalikan x jika x > 0, selain itu 0

Tanpa nonlinearitas seperti ini, network tiga-layer tidak akan bisa memecahkan dataset yang memang tidak punya korelasi langsung, karena secara efektif perilakunya tetap sama dengan network dua-layer biasa.

## 11. Neural Network Dalam Pertama Kita

Mari bangun forward propagation untuk network tiga-layer (dengan satu hidden layer) yang memakai ReLU sebagai nonlinearitas.

In [6]:
import numpy as np
np.random.seed(1)

def relu(x):
    return (x > 0) * x

alpha = 0.2
hidden_size = 4

streetlights = np.array([[1, 0, 1],
                          [0, 1, 1],
                          [0, 0, 1],
                          [1, 1, 1]])

walk_vs_stop = np.array([[1, 1, 0, 0]]).T

# Dua set weight sekarang, untuk menghubungkan tiga layer (inisialisasi acak)
weights_0_1 = 2 * np.random.random((3, hidden_size)) - 1
weights_1_2 = 2 * np.random.random((hidden_size, 1)) - 1

layer_0 = streetlights[0]
layer_1 = relu(np.dot(layer_0, weights_0_1))
layer_2 = np.dot(layer_1, weights_1_2)

print(layer_2)

[0.39194327]


Alur forward propagation-nya: data masuk di `layer_0`, dialirkan ke `layer_1` lewat weighted sum (`weights_0_1`), dilewatkan ke fungsi `relu` (mengubah semua nilai negatif jadi 0), lalu dialirkan sekali lagi ke `layer_2` lewat weighted sum terakhir (`weights_1_2`).

## 12. Backpropagation dalam Kode

Sekarang bagian yang baru: menghitung seberapa besar kontribusi tiap weight terhadap error akhir, termasuk weight di layer pertama (`weights_0_1`).

In [7]:
import numpy as np
np.random.seed(1)

def relu(x):
    return (x > 0) * x

def relu2deriv(output):
    return output > 0  # Mengembalikan 1 untuk input > 0, selain itu 0

streetlights = np.array([[1, 0, 1],
                          [0, 1, 1],
                          [0, 0, 1],
                          [1, 1, 1]])
walk_vs_stop = np.array([[1, 1, 0, 0]]).T

alpha = 0.2
hidden_size = 4

weights_0_1 = 2 * np.random.random((3, hidden_size)) - 1
weights_1_2 = 2 * np.random.random((hidden_size, 1)) - 1

for iteration in range(60):
    layer_2_error = 0
    for i in range(len(streetlights)):
        layer_0 = streetlights[i:i + 1]
        layer_1 = relu(np.dot(layer_0, weights_0_1))
        layer_2 = np.dot(layer_1, weights_1_2)

        layer_2_error += np.sum((layer_2 - walk_vs_stop[i:i + 1]) ** 2)

        layer_2_delta = (walk_vs_stop[i:i + 1] - layer_2)
        layer_1_delta = layer_2_delta.dot(weights_1_2.T) * relu2deriv(layer_1)

        weights_1_2 += alpha * layer_1.T.dot(layer_2_delta)
        weights_0_1 += alpha * layer_0.T.dot(layer_1_delta)

    if iteration % 10 == 9:
        print("Error:" + str(layer_2_error))

# Error akan menurun drastis, dari sekitar 0.63 di awal
# menuju orde 1e-05 setelah 60 iterasi

Error:0.6342311598444467
Error:0.35838407676317513
Error:0.0830183113303298
Error:0.006467054957103705
Error:0.0003292669000750734
Error:1.5055622665134859e-05


### Memahami `layer_1_delta`

Bagian yang benar-benar baru hanyalah cara menghitung delta untuk layer tengah (`layer_1_delta`):

1. Kalikan `layer_2_delta` dengan transpose `weights_1_2`, ini memberi bobot seberapa besar kontribusi tiap node `layer_1` terhadap error di `layer_2` (ini adalah proses backpropagation itu sendiri).
2. Kalikan hasilnya dengan `relu2deriv(layer_1)`, yang bernilai 1 kalau node `layer_1` positif (ikut berkontribusi di forward pass) dan 0 kalau node itu tadi "dimatikan" oleh ReLU. Node yang dimatikan memang tidak berkontribusi ke error, jadi deltanya juga harus dipaksa jadi 0.

Setelah delta untuk `layer_1` didapat, proses mengupdate `weights_0_1` sama persis dengan proses yang sudah dipelajari sebelumnya: kalikan input dengan delta output-nya, lalu gunakan itu untuk mengupdate weight (dengan bantuan alpha).

## 13. Mengapa Deep Network Penting?

Kembali ke analogi klasifikasi gambar kucing: tidak ada satu piksel pun yang berkorelasi langsung dengan "ada kucing atau tidak" dalam sebuah foto. Hanya **kombinasi tertentu** dari banyak piksel (misalnya bentuk telinga, mata, atau bulu kucing) yang berkorelasi.

Inilah esensi deep learning: membangun layer perantara (intermediate layer), di mana setiap node pada layer tersebut merepresentasikan ada/tidaknya suatu **konfigurasi tertentu** dari input. Tidak ada piksel individual yang perlu berkorelasi langsung dengan "kucing", tapi layer tengah bisa belajar mendeteksi konfigurasi seperti "ini mirip telinga kucing" atau "ini mirip mata kucing", dan kombinasi dari sinyal-sinyal ini memberi layer akhir cukup informasi untuk memprediksi dengan akurat.

Prinsip menumpuk layer seperti ini bisa diperluas jauh lebih jauh lagi, beberapa neural network modern punya ratusan layer, masing-masing berperan mendeteksi konfigurasi data yang semakin abstrak dan kompleks dari layer sebelumnya.

## Poin-poin Penting Bab 6

- Neural network pada dasarnya mencari korelasi antara layer input dan layer output, bukan "memahami" data secara literal seperti manusia.
- Error yang dibagi bersama (shared error) antar weight bisa menyebabkan **overfitting**: network berhenti belajar begitu error = 0, meskipun itu terjadi secara kebetulan dan tidak benar-benar generalisasi.
- Kalau tidak ada korelasi langsung antara input dan output, solusinya adalah membangun **dataset perantara** lewat network bertumpuk (hidden layer).
- **Backpropagation** adalah proses menyebarkan sinyal delta dari layer output ke layer-layer sebelumnya, dengan cara mengalikan delta dengan weight yang menghubungkan kedua layer.
- Menumpuk dua layer linear tanpa modifikasi tidak memberi kekuatan tambahan, karena dua weighted sum berurutan bisa selalu disederhanakan jadi satu weighted sum saja.
- **Nonlinearitas** (seperti ReLU: mematikan node yang nilainya negatif) adalah kunci yang memberi hidden layer kemampuan berkorelasi secara kondisional, bukan selalu proporsional terhadap input.
- Kode lengkap backpropagation menggabungkan semua konsep dari bab-bab sebelumnya: forward propagation (predict), penghitungan error dan delta (compare), lalu backpropagation delta ke tiap layer sebelum mengupdate seluruh weight (learn).
- Prinsip menumpuk banyak hidden layer, masing-masing mendeteksi konfigurasi data yang makin kompleks, adalah fondasi dari deep learning modern, dan menjadi topik utama yang akan terus dikembangkan di bab-bab selanjutnya dalam buku ini.

Referensi: Trask, A. W. (2019). *Grokking Deep Learning*, Bab 6 "Building Your First Deep Neural Network: Introduction to Backpropagation". Manning Publications.